# FOLIO First-Order Logic Inference Dataset

This notebook demonstrates the **FOLIO** dataset preparation pipeline.
FOLIO is a natural-language inference (NLI) benchmark where each example has a set of premises and a conclusion labeled **entailment**, **contradiction**, or **neutral**, with gold first-order logic (FOL) annotations.

**Source**: `tasksource/folio` on HuggingFace (mirrors Yale-LILY/FOLIO, Han et al. 2022)

This notebook:
1. Loads a curated mini subset of FOLIO examples
2. Shows the dataset structure and label distribution
3. Demonstrates the data transformation logic (label normalization, example construction)
4. Visualizes dataset statistics

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# No non-standard packages needed — all imports are stdlib or pre-installed on Colab
if 'google.colab' not in sys.modules:
    _pip('matplotlib==3.10.0')

## Imports

All imports used by the original `data.py` script, plus matplotlib for visualization.

In [ ]:
import json
import sys
from pathlib import Path
import matplotlib.pyplot as plt
from collections import Counter

## Data Loading

Load the mini demo dataset. Tries GitHub first (for Colab), falls back to local file.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-8ad782-beam-recall-as-the-binding-constraint-an/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
import os
data = load_data()
print(f"Loaded {len(data['datasets'])} dataset(s)")
print(f"Examples in folio: {len(data['datasets'][0]['examples'])}")

## Config

Tunable parameters for the demo. Set to minimum values for fast execution.
For full reproduction, see the commented original values.

In [ ]:
# Number of examples to process in the demo
N_EXAMPLES = 3  # minimum; original full run: 1204 examples

## Label Normalization

The original FOLIO dataset uses `True`/`False`/`Uncertain` labels. This function maps them to standard NLI labels.

In [ ]:
def normalize_label(label: str) -> str:
    mapping = {"True": "entailment", "False": "contradiction", "Uncertain": "neutral",
               "Entailment": "entailment", "Contradiction": "contradiction", "Neutral": "neutral"}
    return mapping.get(label, label.lower())

## Example Construction

Builds a structured NLI example from a raw FOLIO row. Premises are numbered and formatted as a list; the input field contains the full NLI prompt.

In [ ]:
def make_folio_example(raw: dict, split: str, idx: int) -> dict:
    premises = raw.get("premises", "")
    if isinstance(premises, str):
        premises_list = [p.strip() for p in premises.split("\n") if p.strip()]
    else:
        premises_list = list(premises)

    conclusion = raw.get("conclusion", "")
    label = normalize_label(raw.get("label", ""))
    example_id = raw.get("example_id", idx)
    story_id = raw.get("story_id", "")

    # Input: all premises as numbered list + conclusion question
    premises_text = "\n".join(f"{i+1}. {p}" for i, p in enumerate(premises_list))
    input_text = f"Premises:\n{premises_text}\n\nConclusion: {conclusion}\n\nDoes the conclusion follow from the premises? (entailment/contradiction/neutral)"

    # Output: the label
    output_text = label

    # Metadata
    premises_fol = raw.get("premises-FOL", None)
    conclusion_fol = raw.get("conclusion-FOL", None)

    example = {
        "input": input_text,
        "output": output_text,
        "metadata_example_id": str(example_id),
        "metadata_story_id": str(story_id),
        "metadata_split": split,
        "metadata_row_index": idx,
        "metadata_task_type": "classification",
        "metadata_n_classes": 3,
        "metadata_conclusion": conclusion,
        "metadata_num_premises": len(premises_list),
    }

    if premises_fol:
        example["metadata_premises_fol"] = str(premises_fol)
    if conclusion_fol:
        example["metadata_conclusion_fol"] = str(conclusion_fol)

    return example

## Dataset Processing

Load examples from the pre-built dataset (loaded from mini_demo_data.json) and inspect the structure.

In [ ]:
# Extract examples from loaded data
all_examples = data["datasets"][0]["examples"][:N_EXAMPLES]
print(f"Processing {len(all_examples)} examples")

# Show label distribution
label_counts = Counter(ex["output"] for ex in all_examples)
print("\nLabel distribution:")
for label, count in sorted(label_counts.items()):
    print(f"  {label}: {count}")

## Inspect a Sample Example

Look at the first example to understand the full NLI input/output structure.

In [ ]:
ex = all_examples[0]
print("=== Example ===")
print(f"ID: {ex['metadata_example_id']}")
print(f"Split: {ex['metadata_split']}")
print(f"Num premises: {ex['metadata_num_premises']}")
print(f"Label: {ex['output']}")
print()
print("--- Input (truncated) ---")
print(ex['input'][:500], "...")
print()
print("--- Conclusion ---")
print(ex['metadata_conclusion'])
print()
if 'metadata_premises_fol' in ex:
    print("--- FOL premises (truncated) ---")
    print(ex['metadata_premises_fol'][:300], "...")

## Output Structure

The full pipeline outputs a JSON with a `datasets` key containing all examples. Reconstruct the output format here.

In [ ]:
output = {
    "datasets": [
        {
            "dataset": "folio",
            "examples": all_examples,
        }
    ]
}

print(f"Output structure:")
print(f"  datasets: {len(output['datasets'])}")
print(f"  examples: {len(output['datasets'][0]['examples'])}")
print(f"  keys per example: {list(output['datasets'][0]['examples'][0].keys())}")

## Visualization

Summary statistics and label distribution chart.

In [ ]:
# Summary table
print("=" * 50)
print("FOLIO Demo Dataset Summary")
print("=" * 50)
print(f"Total examples processed : {len(all_examples)}")
print(f"Task type                : {all_examples[0]['metadata_task_type']}")
print(f"Num classes              : {all_examples[0]['metadata_n_classes']}")
print()
print("Label Distribution:")
for label, count in sorted(label_counts.items()):
    bar = '#' * count
    print(f"  {label:<15} {count:>3}  {bar}")
print()
print("Premise counts:")
premise_counts = [ex['metadata_num_premises'] for ex in all_examples]
print(f"  min={min(premise_counts)}, max={max(premise_counts)}, avg={sum(premise_counts)/len(premise_counts):.1f}")

# Bar chart of label distribution
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

labels = list(label_counts.keys())
counts = list(label_counts.values())
colors = {'entailment': '#2ecc71', 'contradiction': '#e74c3c', 'neutral': '#3498db'}
bar_colors = [colors.get(l, '#95a5a6') for l in labels]
axes[0].bar(labels, counts, color=bar_colors)
axes[0].set_title('Label Distribution')
axes[0].set_ylabel('Count')
axes[0].set_xlabel('Label')

# Premises per example
splits = [ex['metadata_split'] for ex in all_examples]
split_counter = Counter(splits)
axes[1].bar(list(split_counter.keys()), list(split_counter.values()), color='#9b59b6')
axes[1].set_title('Examples by Split')
axes[1].set_ylabel('Count')
axes[1].set_xlabel('Split')

plt.tight_layout()
plt.savefig('folio_demo_stats.png', dpi=80, bbox_inches='tight')
plt.show()
print("Plot saved to folio_demo_stats.png")